# Replay Recorded Runs (No Docker, No AWS)

This notebook grades and examines real agent runs that we recorded. You do not need Docker, Harbor, AWS credentials, or tokens. It costs nothing, and it runs in a minute.

Use it when:

- You want to learn how to read the results before you set up Docker and Bedrock (notebook 01).
- You present the module and do not have time for a live run.
- You use the interactive tutor (`Interactive Learning/workload evals/SKILL-coding-assistant.md`) and want real outputs to discuss.

The recordings are in `datasets/recorded-runs/`. Each one is a Harbor job of notebook 02 or 03, with the same files that you get when you run it yourself: the agent's trajectory, its findings or patch, and every file of the verifier. To make your own recordings, see section 5.

### Procedure

1. See what was recorded.
2. PR review: grade the reviews of two agents, and find why one missed a security defect.
3. Coding: read the reward stack of each fix.
4. Compare the agents.
5. Record your own runs.

In [ ]:
import json
import tomllib
from pathlib import Path

import coding_eval as ce
from coding_eval import read_json, read_toml

RECORDED = ce.RECORDED_DIR
PR_TASKS = Path("datasets/awslabs-mcp-pr-review/tasks")
CODING_TASKS = Path("datasets/strands-evals-coding/tasks")

def trials(kind: str):
    """Each recorded trial of one kind ("pr-review" or "coding"), as (agent, trial directory)."""
    for agent_dir in sorted(p for p in RECORDED.iterdir() if p.is_dir()):
        for trial in sorted((agent_dir / kind).glob("*/result.json")):
            yield agent_dir.name, trial.parent

def task_of(trial: Path) -> str:
    return read_json(trial / "result.json")["task_name"].replace("/", "__")

def agent_commands(trial: Path) -> list[str]:
    """The shell commands that the agent ran, from its trajectory."""
    path = trial / "agent" / "trajectory.json"
    steps = read_json(path).get("steps", []) if path.exists() else []
    commands = []
    for step in steps:
        for call in step.get("tool_calls") or []:
            args = call.get("arguments") or {}
            commands.append(str(args.get("command") or args.get("cmd") or args)[:160])
    return commands

def final_message(trial: Path) -> str:
    path = trial / "agent" / "trajectory.json"
    steps = read_json(path).get("steps", []) if path.exists() else []
    texts = [s.get("message") for s in steps if s.get("source") == "agent" and isinstance(s.get("message"), str) and s["message"].strip()]
    return texts[-1] if texts else ""

## 1. What Was Recorded

`README.md` in `datasets/recorded-runs/` says when and how the runs were made. Each row below is one trial: one agent on one task.

In [ ]:
print((RECORDED / "README.md").read_text())
for kind in ("pr-review", "coding"):
    for agent, trial in trials(kind):
        result = read_json(trial / "result.json")
        info, usage = result.get("agent_info") or {}, result.get("agent_result") or {}
        model = (info.get("model_info") or {}).get("name")
        print(f"{kind:<9} {agent:<12} {task_of(trial):<36} model={model}  "
              f"input tokens={usage.get('n_input_tokens')}  cost=${(usage.get('cost_usd') or 0):.2f}")

## 2. PR Review: Grade the Reviews

The verifier has already scored each review (notebook 02, section 4). Here you apply the pass/fail rule of notebook 02 to the recorded reward files:

- **Approve task:** pass if the agent approves (no `high` finding).
- **Reject task:** pass if the agent blocks (a `high` finding) and finds every high-severity oracle finding (`coverage_required` is 1.0).

In [ ]:
def review_row(trial: Path) -> dict:
    task = task_of(trial)
    harvest = read_toml(PR_TASKS / task / "task.toml")["metadata"]["harvest"]
    path = trial / "artifacts" / "logs" / "artifacts" / "findings.json"
    findings = read_json(path) if path.exists() else []
    predicted = "block" if any(f.get("severity") == "high" for f in findings) else "approve"
    reward = read_json(trial / "verifier" / "reward.json")
    ok = predicted == harvest["expected_verdict"] and (predicted == "approve" or reward.get("coverage_required") == 1.0)
    return {"task": task, "expected": harvest["expected_verdict"], "predicted": predicted, "passed": ok,
            "findings": findings, **reward}

for agent, trial in trials("pr-review"):
    r = review_row(trial)
    print(f"{'PASS' if r['passed'] else 'FAIL'}  {agent:<12} {r['task']}: expected {r['expected']}, predicted {r['predicted']}, "
          f"coverage_required={r.get('coverage_required')}, coverage_all={r.get('coverage_all'):.2f}")
    for f in r["findings"]:
        print(f"        [{f.get('severity')}] {f.get('path', '').split('/')[-1]}:{f.get('line')} {f.get('statement', '')[:110]}")

### Why did an agent miss the defect?

The score tells you that a review failed. The trajectory tells you why. The oracle finding that blocks PR 4615 is below. Before you run the next cell, think: what would a reviewer have to *try* to find this defect?

In [ ]:
oracle = read_json(PR_TASKS / "awslabs__mcp__pr4615-reject" / "tests" / "oracle.json")
high = next(f for f in oracle["findings"] if f["severity"] == "high")
print("Oracle (high):", high["statement"], "\n")

for agent, trial in trials("pr-review"):
    if task_of(trial) != "awslabs__mcp__pr4615-reject":
        continue
    r = review_row(trial)
    commands = agent_commands(trial)
    print(f"== {agent}: {'PASS' if r['passed'] else 'FAIL'}, {len(commands)} commands")
    for c in commands:
        print("   $", c.replace("\n", " ")[:150])
    print("   Final message:", final_message(trial)[:400].replace("\n", " "), "\n")

**Questions to answer:**

1. Did the agent that failed ever run the guard with an input that starts with a comment? Which commands show what it tried?
2. All tests passed. Why is a passing test suite not evidence that the guard is safe?
3. Which cause from notebook 02, section 6, fits this failure: the agent misses a defect that it can find, the agent does not have the reviewer's context, or the datapoint is wrong?

## 3. Coding: Read the Reward Stack

A fix is a **tasteful solve** only if every gate passes (notebook 03, section 4). `correct` covers the first two gates: the change works. The rest ask if a maintainer would merge it. Every LLM gate is built from yes/no checks.

In [ ]:
def show_fix(agent: str, trial: Path) -> dict:
    details = read_json(trial / "verifier" / "reward_details.json")
    flat, checks = details.get("flat", {}), details.get("checks", {})
    status = "INVALID" if details.get("invalid") else ("TASTEFUL SOLVE" if flat.get("reward") == 1.0 else
                                                      "CORRECT, NOT TASTEFUL" if flat.get("correct") == 1.0 else "FAIL")
    print(f"== {status}  {agent}  {task_of(trial)}")
    for gate, ok in checks.items():
        print(f"     {'pass' if ok else 'FAIL'}  {gate}")
    for line in details.get("failed_tests", [])[:5]:
        print(f"     failed test: {line}")
    for name, check in (details.get("taste", {}).get("dimensions") or {}).items():
        if isinstance(check, dict) and check.get("passed") is False:
            print(f"     taste check failed: {name}: {check.get('reason', '')[:140]}")
    for c in details.get("rubric", {}).get("criteria", []):
        if c.get("score") == 0:
            print(f"     rubric criterion failed: {c['id']}: {c.get('reason', '')[:140]}")
    print()
    return flat

for agent, trial in trials("coding"):
    show_fix(agent, trial)

### Compare a fix with the merged fix

Patch bloat compares the size of the agent's change with the merged fix. Read both for one trial: the first one that is not a tasteful solve, or else the first trial.


In [ ]:
coding_trials = list(trials("coding"))
not_solved = [(a, t) for a, t in coding_trials if read_json(t / "verifier" / "reward_details.json").get("flat", {}).get("reward") != 1.0]
agent, trial = (not_solved or coding_trials)[0]
details = read_json(trial / "verifier" / "reward_details.json")
task = task_of(trial)
print(f"== {agent} on {task}: reward={details['flat'].get('reward')} patch_bloat={details['flat'].get('patch_bloat')}")
print((trial / "verifier" / "agent.patch").read_text()[:2500])
print("\n-- merged fix --")
print((CODING_TASKS / task / "solution" / "oracle.patch").read_text()[:2500])


**Questions to answer:**

1. If a fix fails a hidden test (in your own runs, or in an earlier recording): did the agent write a test that reproduces the bug (`agent_tests_reproduce_bug`)? What case did its test miss?
2. For a fix that is correct but not tasteful: do you agree with each failed taste check? If not, which word in the check would you change?
3. Why does the rubric not gate a solve alone? (Hint: what does an empty change get on a "does not change the interface" criterion?)

## 4. Compare the Agents

The same tasks and the same grader, with two agents. With one trial for each task, a difference can be variance. Before you decide which agent is better, run more trials (`N_ATTEMPTS` in notebooks 02 and 03).

In [ ]:
summary: dict[str, dict] = {}
for agent, trial in trials("pr-review"):
    s = summary.setdefault(agent, {"reviews passed": 0, "reviews": 0, "correct fixes": 0, "tasteful solves": 0, "fixes": 0, "cost": 0.0})
    s["reviews"] += 1; s["reviews passed"] += review_row(trial)["passed"]
    s["cost"] += (read_json(trial / "result.json").get("agent_result") or {}).get("cost_usd") or 0
for agent, trial in trials("coding"):
    s = summary.setdefault(agent, {"reviews passed": 0, "reviews": 0, "correct fixes": 0, "tasteful solves": 0, "fixes": 0, "cost": 0.0})
    flat = read_json(trial / "verifier" / "reward_details.json").get("flat", {})
    s["fixes"] += 1; s["correct fixes"] += flat.get("correct") == 1.0; s["tasteful solves"] += flat.get("reward") == 1.0
    s["cost"] += (read_json(trial / "result.json").get("agent_result") or {}).get("cost_usd") or 0
for agent, s in summary.items():
    print(f"{agent:<12} reviews {s['reviews passed']}/{s['reviews']} · correct fixes {s['correct fixes']}/{s['fixes']} · "
          f"tasteful solves {s['tasteful solves']}/{s['fixes']} · agent cost ${s['cost']:.2f}")

## 5. Record Your Own Runs

After you run notebook 02 or 03, record a job so that other people can replay it:

```python
import coding_eval as ce
ce.record_job("jobs/codex", ce.RECORDED_DIR / "my-agent" / "pr-review")
```

`record_job` keeps only the files of this notebook, and keeps only the result fields of each trial, not the job configuration. It refuses to write anything, and tells you the file, if a file looks like it has an AWS access key, an ARN with an account ID, or a local home path. Examine the trajectory yourself too before you share a recording: an agent can print anything that it reads.

## Summary

- You can learn to read coding-agent results without Docker or AWS.
- A passing test suite is not evidence that a review is correct: read the trajectory to see what the agent tried.
- A fix must work *and* be one that a maintainer would merge. Each gate is a yes/no check.
- One trial is not enough to compare agents.

**Next:** To run the agents yourself, do the setup in [01 intro and setup](01%20intro%20and%20setup.ipynb), then [02 PR agent eval](02%20PR%20agent%20eval.ipynb) and [03 coding agent eval](03%20coding%20agent%20eval.ipynb).